In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load data
file_path = "../Data/Raw/telecom_customer_segmentation.xlsx"
df = pd.read_excel(file_path, engine='openpyxl')

print("Original shape:", df.shape)
df.head()

Original shape: (5000, 20)


,customer_id,age,gender,region,plan_type,tenure_months,monthly_revenue_usd,total_call_minutes,data_usage_gb,sms_count,intl_call_minutes,roaming_enabled,logins_per_month,support_tickets_6mo,days_since_last_call,num_services_subscribed,auto_pay_enrolled,paperless_billing,contract_type,true_segment
0,CUST00001,51,Female,East,Premium,54,47.52,302.2,2.83,97,6,0,13,0,43,4,1,1,2-Year,Moderate User
1,CUST00002,59,Female,North,Basic,15,39.73,127.6,2.44,68,5,0,5,2,46,1,0,1,Month-to-Month,Churner
2,CUST00003,67,Female,South,Standard,3,24.09,76.3,1.76,10,1,0,8,0,43,2,0,1,1-Year,Low User
3,CUST00004,55,Female,West,Basic,17,26.43,12.4,2.19,40,2,0,1,0,42,3,0,0,2-Year,Low User
4,CUST00005,19,Female,North,Unlimited,25,118.19,955.0,30.06,320,19,0,24,2,51,5,0,0,2-Year,Heavy User


In [3]:
# Drop customer_id and true_segment (target for clustering is not needed)
df_clean = df.drop(columns=['customer_id', 'true_segment'])

print("Columns after dropping:", df_clean.columns.tolist())
print("New shape:", df_clean.shape)

Columns after dropping: ['age', 'gender', 'region', 'plan_type', 'tenure_months', 'monthly_revenue_usd', 'total_call_minutes', 'data_usage_gb', 'sms_count', 'intl_call_minutes', 'roaming_enabled', 'logins_per_month', 'support_tickets_6mo', 'days_since_last_call', 'num_services_subscribed', 'auto_pay_enrolled', 'paperless_billing', 'contract_type']
New shape: (5000, 18)


In [4]:
# Separate numerical and categorical columns
numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['age', 'tenure_months', 'monthly_revenue_usd', 'total_call_minutes', 'data_usage_gb', 'sms_count', 'intl_call_minutes', 'roaming_enabled', 'logins_per_month', 'support_tickets_6mo', 'days_since_last_call', 'num_services_subscribed', 'auto_pay_enrolled', 'paperless_billing']
Categorical columns: ['gender', 'region', 'plan_type', 'contract_type']


C:\Users\Arosha IIT\AppData\Local\Temp\ipykernel_30016\1507829930.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()


In [5]:
# Label Encoding for plan_type, contract_type, gender
label_encoders = {}
for col in ['plan_type', 'contract_type', 'gender']:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f"Mapping for {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# One-Hot Encoding for region (drop first to avoid multicollinearity)
df_clean = pd.get_dummies(df_clean, columns=['region'], drop_first=True)

print("\nAfter encoding, all columns are numeric.")
print("Column types:\n", df_clean.dtypes.value_counts())

Mapping for plan_type: {'Basic': np.int64(0), 'Business': np.int64(1), 'Premium': np.int64(2), 'Standard': np.int64(3), 'Unlimited': np.int64(4)}
Mapping for contract_type: {'1-Year': np.int64(0), '2-Year': np.int64(1), 'Month-to-Month': np.int64(2)}
Mapping for gender: {'Female': np.int64(0), 'Male': np.int64(1)}

After encoding, all columns are numeric.
Column types:
 int64      14
bool        4
float64     3
Name: count, dtype: int64


In [6]:
# All columns are now numeric
feature_columns = df_clean.columns.tolist()

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_clean),
    columns=feature_columns
)

print("Scaled data shape:", df_scaled.shape)
print("\nFirst 5 rows after scaling:")
df_scaled.head()

Scaled data shape: (5000, 21)

First 5 rows after scaling:


,age,gender,plan_type,tenure_months,monthly_revenue_usd,total_call_minutes,data_usage_gb,sms_count,intl_call_minutes,roaming_enabled,...,support_tickets_6mo,days_since_last_call,num_services_subscribed,auto_pay_enrolled,paperless_billing,contract_type,region_East,region_North,region_South,region_West
0,0.282710,-0.972771,-0.096664,0.433507,-0.277783,-0.289662,-0.972645,-0.414478,-0.545286,-0.841913,...,-0.656999,0.771235,0.708594,0.721262,0.652161,-0.156360,1.980217,-0.497499,-0.509047,-0.487467
1,0.771681,-0.972771,-1.579246,-0.762349,-0.410246,-0.912136,-1.016143,-0.632266,-0.590714,-0.841913,...,0.704657,0.943961,-1.415489,-1.386459,0.652161,1.010505,-0.504995,2.010055,-0.509047,-0.487467
2,1.260651,-0.972771,0.644627,-1.130305,-0.676191,-1.095028,-1.091987,-1.067840,-0.772428,-0.841913,...,-0.656999,0.771235,-0.707462,-1.386459,0.652161,-1.323225,-0.504995,-0.497499,1.964456,-0.487467
3,0.527196,-0.972771,-1.579246,-0.701023,-0.636401,-1.322840,-1.044027,-0.842543,-0.726999,-0.841913,...,-0.656999,0.713660,0.000566,-1.386459,-1.533364,-0.156360,-0.504995,-0.497499,-0.509047,2.051422
4,-1.673171,-0.972771,1.385917,-0.455719,0.923901,2.037663,2.064445,1.260232,0.045283,-0.841913,...,0.704657,1.231839,1.416622,-1.386459,-1.533364,-0.156360,-0.504995,2.010055,-0.509047,-0.487467


In [7]:
import os

# Ensure the output directory exists
output_dir = "../Data/Preprocessed"
os.makedirs(output_dir, exist_ok=True)

# Save as CSV
output_path = os.path.join(output_dir, "Preprocessed.csv")
df_scaled.to_csv(output_path, index=False)

print(f"Preprocessed data saved to: {output_path}")
print(f"Final shape: {df_scaled.shape}")

Preprocessed data saved to: ../Data/Preprocessed\Preprocessed.csv
Final shape: (5000, 21)
